In [1]:
!pip install fastapi uvicorn nest-asyncio pydantic groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.8 MB/s eta 0:00:00


In [2]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional
import nest_asyncio
import uvicorn
import threading
import json
import os
from groq import Groq
from google.colab import userdata

nest_asyncio.apply()

client_groq = Groq(api_key=userdata.get('GROQ_API_KEY'))
app = FastAPI(title="Microsoft Graph AI Integration")

print("✅ Setup complete")

✅ Setup complete


In [3]:
def check_customer_risk(balance: float, transactions: int, is_active: bool) -> str:
    if not is_active:
        return "HIGH RISK — Account inactive"
    if balance < 1000:
        return f"HIGH RISK — Low balance: €{balance}"
    elif balance < 5000:
        return f"MEDIUM RISK — Moderate balance: €{balance}"
    else:
        return f"LOW RISK — Healthy balance: €{balance}"

def check_aml_flag(transaction_amount: float, country: str) -> str:
    high_risk_countries = ["Country_A", "Country_B", "Country_C"]
    flags = []
    if transaction_amount > 10000:
        flags.append("Transaction above EUR 10,000 — monitoring required")
    if country in high_risk_countries:
        flags.append(f"{country} is high risk — Enhanced Due Diligence required")
    if flags:
        return "🚨 AML FLAGS:\n" + "\n".join(flags)
    return "✅ No AML flags"

def check_credit_limit(client_exposure: float, total_portfolio: float, sector: str) -> str:
    exposure_pct = (client_exposure / total_portfolio) * 100
    issues = []
    if exposure_pct > 10:
        issues.append(f"⚠️ Exposure {exposure_pct:.1f}% exceeds 10% limit")
    if client_exposure > 5000000:
        issues.append("⚠️ Above EUR 5M — immediate escalation required")
    if issues:
        return "LIMIT BREACH:\n" + "\n".join(issues)
    return f"✅ Exposure {exposure_pct:.1f}% — within limits"

def run_compliance_check(message: str) -> str:
    """Send message to AI and get compliance assessment"""
    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a banking compliance officer.
                Extract any client details from the message and provide
                a brief compliance assessment.
                Be concise — this is a Teams message response."""
            },
            {
                "role": "user",
                "content": message
            }
        ]
    )
    return response.choices[0].message.content

print("✅ Compliance tools ready")

✅ Compliance tools ready


In [4]:
# This is what a real Microsoft Teams message looks like
# when received via Microsoft Graph API webhook

class TeamsMessage(BaseModel):
    message_id: str
    from_user: str
    team_name: str
    channel: str
    content: str
    timestamp: str

class TeamsResponse(BaseModel):
    status: str
    original_message: str
    ai_response: str
    compliance_flags: list
    sent_to: str

print("✅ Teams message models defined")
print("""
Real Microsoft Graph webhook payload looks like:
{
    "value": [{
        "changeType": "created",
        "resource": "teams/{teamId}/channels/{channelId}/messages/{messageId}",
        "resourceData": {
            "body": {"content": "Check client balance €3500"},
            "from": {"user": {"displayName": "John Smith"}}
        }
    }]
}

We simulate this structure today.
Real implementation just swaps the data source.
""")

✅ Teams message models defined

Real Microsoft Graph webhook payload looks like:
{
    "value": [{
        "changeType": "created",
        "resource": "teams/{teamId}/channels/{channelId}/messages/{messageId}",
        "resourceData": {
            "body": {"content": "Check client balance €3500"},
            "from": {"user": {"displayName": "John Smith"}}
        }
    }]
}

We simulate this structure today.
Real implementation just swaps the data source.



In [5]:
# Simulate Microsoft Graph webhook verification
# Real Graph API sends this to verify your endpoint
@app.get("/api/teams/webhook")
def verify_webhook(validationToken: str = None):
    """
    Microsoft Graph sends a validation token
    when you register a webhook.
    You must return it to prove you own the endpoint.
    """
    if validationToken:
        print(f"✅ Webhook verified with token: {validationToken[:20]}...")
        return validationToken
    return {"status": "webhook endpoint active"}


# Simulate receiving a Teams message
@app.post("/api/teams/message")
def receive_teams_message(message: TeamsMessage):
    """
    In real Microsoft Graph this endpoint receives
    notifications when someone posts in Teams.

    Real flow:
    User posts in Teams
        → Microsoft Graph sends webhook to this URL
        → We process the message
        → We call Graph API to reply
    """
    print(f"📨 Message received from {message.from_user}")
    print(f"📍 In: {message.team_name} → {message.channel}")
    print(f"💬 Content: {message.content}")

    # Run AI compliance check
    ai_response = run_compliance_check(message.content)

    # Extract any compliance flags
    flags = []
    if "HIGH RISK" in ai_response:
        flags.append("HIGH_RISK_CLIENT")
    if "AML" in ai_response.upper():
        flags.append("AML_FLAG")
    if "BREACH" in ai_response.upper():
        flags.append("LIMIT_BREACH")
    if "ESCALAT" in ai_response.upper():
        flags.append("ESCALATION_REQUIRED")

    return TeamsResponse(
        status="success",
        original_message=message.content,
        ai_response=ai_response,
        compliance_flags=flags,
        sent_to=f"{message.team_name}/{message.channel}"
    )


# Simulate sending a proactive Teams message
@app.post("/api/teams/alert")
def send_teams_alert(
    team: str = "Compliance Team",
    channel: str = "General",
    alert_type: str = "HIGH_RISK",
    client_name: str = "Unknown Client"
):
    """
    In real Microsoft Graph you'd call:
    POST https://graph.microsoft.com/v1.0/teams/{teamId}/channels/{channelId}/messages

    With your OAuth2 token to send proactive alerts.
    """

    alert_messages = {
        "HIGH_RISK": f"🚨 HIGH RISK ALERT: {client_name} flagged for immediate review",
        "AML_FLAG": f"⚠️ AML FLAG: {client_name} transaction requires investigation",
        "LIMIT_BREACH": f"📊 LIMIT BREACH: {client_name} exposure exceeds policy limits",
        "ESCALATION": f"⬆️ ESCALATION: {client_name} requires senior management review"
    }

    message = alert_messages.get(alert_type, f"Alert for {client_name}")

    return {
        "status": "simulated",
        "would_send_to": f"{team}/{channel}",
        "message": message,
        "real_graph_endpoint": f"POST https://graph.microsoft.com/v1.0/teams/{{teamId}}/channels/{{channelId}}/messages",
        "note": "In production replace simulation with real Graph API call using OAuth2 token"
    }


# Dashboard — show all available endpoints
@app.get("/")
def home():
    return {
        "service": "Northern Europe Bank — AI Teams Integration",
        "version": "1.0",
        "endpoints": {
            "GET  /api/teams/webhook": "Microsoft Graph webhook verification",
            "POST /api/teams/message": "Receive and process Teams messages",
            "POST /api/teams/alert":   "Send proactive compliance alerts to Teams",
            "GET  /docs":              "Full API documentation"
        },
        "real_graph_endpoints": {
            "auth":     "POST https://login.microsoftonline.com/{tenantId}/oauth2/v2.0/token",
            "send":     "POST https://graph.microsoft.com/v1.0/teams/{teamId}/channels/{channelId}/messages",
            "receive":  "Webhook subscription via Graph API"
        }
    }

print("✅ All endpoints defined")
print("   GET  /api/teams/webhook — webhook verification")
print("   POST /api/teams/message — receive Teams messages")
print("   POST /api/teams/alert   — send Teams alerts")
print("   GET  /                  — service dashboard")

✅ All endpoints defined
   GET  /api/teams/webhook — webhook verification
   POST /api/teams/message — receive Teams messages
   POST /api/teams/alert   — send Teams alerts
   GET  /                  — service dashboard


In [6]:
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8001)

thread = threading.Thread(target=run_server)
thread.daemon = True
thread.start()

print("✅ Server running on port 8001")

✅ Server running on port 8001


In [7]:
import requests

base_url = "http://localhost:8001"

# Test 1 — Home dashboard
print("TEST 1 — Dashboard")
print("=" * 55)
r = requests.get(f"{base_url}/")
print(json.dumps(r.json(), indent=2))

# Test 2 — Simulate Teams message
print("\nTEST 2 — Teams Message")
print("=" * 55)
r = requests.post(f"{base_url}/api/teams/message", json={
    "message_id": "msg_001",
    "from_user": "John Smith",
    "team_name": "Compliance Team",
    "channel": "Client Reviews",
    "content": "Please check new client: balance €750, 89 transactions, active, transaction €15000 from Country_A",
    "timestamp": "2024-07-19T10:30:00Z"
})
print(json.dumps(r.json(), indent=2))

# Test 3 — Send Teams alert
print("\nTEST 3 — Teams Alert")
print("=" * 55)
r = requests.post(
    f"{base_url}/api/teams/alert",
    params={
        "team": "Compliance Team",
        "channel": "Alerts",
        "alert_type": "HIGH_RISK",
        "client_name": "Jan de Vries"
    }
)
print(json.dumps(r.json(), indent=2))

TEST 1 — Dashboard
INFO:     127.0.0.1:54526 - "GET / HTTP/1.1" 200 OK
{
  "service": "Northern Europe Bank \u2014 AI Teams Integration",
  "version": "1.0",
  "endpoints": {
    "GET  /api/teams/webhook": "Microsoft Graph webhook verification",
    "POST /api/teams/message": "Receive and process Teams messages",
    "POST /api/teams/alert": "Send proactive compliance alerts to Teams",
    "GET  /docs": "Full API documentation"
  },
  "real_graph_endpoints": {
    "auth": "POST https://login.microsoftonline.com/{tenantId}/oauth2/v2.0/token",
    "send": "POST https://graph.microsoft.com/v1.0/teams/{teamId}/channels/{channelId}/messages",
    "receive": "Webhook subscription via Graph API"
  }
}

TEST 2 — Teams Message
📨 Message received from John Smith
📍 In: Compliance Team → Client Reviews
💬 Content: Please check new client: balance €750, 89 transactions, active, transaction €15000 from Country_A
INFO:     127.0.0.1:54534 - "POST /api/teams/message HTTP/1.1" 200 OK
{
  "status": "succ